In [1]:
# Libraries
import yfinance as yf
import ta

import pandas as pd 
import numpy as np

import matplotlib.pyplot as plt
plt.style.use('default')
%matplotlib inline
from mplfinance.original_flavor import candlestick_ohlc
import matplotlib.dates as mpdates

from tensorflow.keras.layers import Input, LSTM, Dense, concatenate
from tensorflow.keras.models import Model


In [2]:
import tensorflow as tf
print(tf.__version__)
from tensorflow.keras.layers import Input, LSTM, Dense, concatenate



2.19.0


In [3]:
# Load dataset
df_KLCI = pd.read_csv('merged_KLCI_sentiments.csv', parse_dates=['Date'])
df_index = df_KLCI.set_index('Date').loc['2019-01-01':'2025-01-01'].copy()
df_KLCI

,Date,Close,High,Low,Open,Volume,headline,Polarity,Subjectivity,Sentiment,VADER_Score,VADER_Sentiment
0,2019-01-02,1668.109985,1694.099976,1666.069946,1693.510010,47769800,NaN,0.00,0.0,Neutral,0.0000,Neutral
1,2019-01-03,1675.829956,1680.780029,1666.219971,1675.160034,69484500,NaN,0.00,0.0,Neutral,0.0000,Neutral
2,2019-01-04,1669.780029,1676.550049,1668.150024,1672.579956,64869000,NaN,0.00,0.0,Neutral,0.0000,Neutral
3,2019-01-07,1679.170044,1687.130005,1673.030029,1674.920044,144668500,NaN,0.00,0.0,Neutral,0.0000,Neutral
4,2019-01-08,1672.760010,1686.369995,1670.670044,1685.719971,176071900,NaN,0.00,0.0,Neutral,0.0000,Neutral
...,...,...,...,...,...,...,...,...,...,...,...,...
1465,2024-12-24,1602.989990,1603.890015,1596.640015,1596.640015,95048400,NaN,0.00,0.0,Neutral,0.0000,Neutral
1466,2024-12-26,1613.699951,1615.280029,1602.790039,1603.579956,120934600,Skin Tightening and Lifting Treatment - MH Clinic,0.00,0.0,Neutral,0.0000,Neutral
1467,2024-12-27,1628.140015,1632.630005,1616.689941,1617.560059,130614200,"Military Connections, Investment Efficiency an...",-0.05,0.1,Neutral,0.0258,Neutral
1468,2024-12-30,1637.680054,1638.560059,1624.920044,1626.140015,134711300,NaN,0.00,0.0,Neutral,0.0000,Neutral


In [4]:
# Calculate MACD
df_index['MACD']        = ta.trend.macd(df_index['Close'])
df_index['MACD_signal'] = ta.trend.macd_signal(df_index['Close'])
df_index['MACD_hist']   = ta.trend.macd_diff(df_index['Close'])

In [5]:
# RSI for 14 days
df_index['RSI_14'] = ta.momentum.rsi(df_index['Close'])

In [6]:
# Moving averages
df_index['SMA_13'] = ta.trend.sma_indicator(df_index['Close'], window=13)
df_index['SMA_49'] = ta.trend.sma_indicator(df_index['Close'], window=49)

df_index['EMA_13'] = ta.trend.ema_indicator(df_index['Close'], window=13)
df_index['EMA_49'] = ta.trend.ema_indicator(df_index['Close'], window=49)

In [7]:
df_index.describe()

,Close,High,Low,Open,Volume,Polarity,Subjectivity,VADER_Score,MACD,MACD_signal,MACD_hist,RSI_14,SMA_13,SMA_49,EMA_13,EMA_49
count,1470.000000,1470.000000,1470.000000,1470.000000,1.470000e+03,1470.000000,1470.000000,1470.000000,1445.000000,1437.000000,1437.000000,1457.000000,1458.000000,1422.000000,1458.000000,1422.000000
mean,1540.670361,1546.599885,1534.126762,1540.821660,1.652938e+08,0.005336,0.019974,0.015663,-0.354017,-0.366358,-0.014453,49.639258,1539.809243,1537.069102,1539.806728,1536.891937
std,84.411991,84.351340,84.722369,84.981869,8.865963e+07,0.058226,0.097936,0.118993,12.755448,11.937531,4.101899,11.891170,82.413627,76.287698,81.543649,73.610207
min,1219.719971,1242.819946,1207.800049,1217.280029,0.000000e+00,-0.400000,0.000000,-0.648600,-72.497558,-60.095010,-24.460217,12.630243,1296.026921,1360.777149,1328.307343,1394.441868
25%,1474.102509,1478.997528,1466.795013,1473.595032,1.057776e+08,0.000000,0.000000,0.000000,-7.872606,-7.514265,-2.409811,41.033851,1474.733274,1468.538263,1473.787382,1471.295108
50%,1553.304993,1558.184998,1545.090027,1552.765015,1.451446e+08,0.000000,0.000000,0.000000,-0.073198,-0.042356,-0.143402,49.748254,1547.919236,1547.532858,1550.310794,1546.116868
75%,1602.704987,1608.949982,1595.967529,1603.204956,2.016254e+08,0.000000,0.000000,0.000000,6.943219,6.453052,2.142096,58.044466,1600.265186,1596.051631,1600.848389,1594.003702
max,1730.680054,1732.270020,1719.670044,1730.510010,9.730662e+08,1.000000,1.000000,0.918600,47.667706,39.025311,14.574670,80.941341,1708.844623,1690.555923,1709.285852,1687.844651
